# FTR-202 — Qwen3.8-27B teacher benchmark generation

This notebook runs **generation only** for the precommitted FTR-202 benchmark on a Google Colab **A100 80 GB** runtime.

Immutable execution source:

`90d42cca541cc7a96493d5d80d95718e5930d551`

Pinned teacher:

`Qwen/Qwen3.8-27B@72a217afab8029b39e4af1c7273a829995a3dbaf`

The notebook deliberately never executes generated benchmark code. Checkpoints and the final transport manifest are written directly to Google Drive so an interrupted Colab session can resume.


In [ ]:
!nvidia-smi


In [ ]:
from pathlib import Path
import subprocess

SOURCE_SHA = "90d42cca541cc7a96493d5d80d95718e5930d551"
REPO = Path("/content/tiny-qwen-coder")
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/ekkus93/tiny-qwen-coder.git", str(REPO)],
        check=True,
    )

subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_SHA], check=True)

observed = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
dirty = subprocess.check_output(
    ["git", "-C", str(REPO), "status", "--porcelain"], text=True
).strip()
assert observed == SOURCE_SHA, (observed, SOURCE_SHA)
assert not dirty, dirty
print("source_sha=", observed)


In [ ]:
%cd /content/tiny-qwen-coder
!python -m pip install -q "uv==0.12.13"
!uv sync --frozen


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path

PERSISTENT_ROOT = Path("/content/drive/MyDrive/tiny-qwen-coder/ftr-202")
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
print(PERSISTENT_ROOT)


In [ ]:
import subprocess

subprocess.run(
    [
        "uv", "run", "--frozen", "python", "-m",
        "tiny_qwen_coder.evaluation.python_ftr_teacher_superiority",
        "audit",
        "--repo-root", ".",
        "--source-git-sha", SOURCE_SHA,
    ],
    check=True,
)


In [ ]:
import subprocess

subprocess.run(
    [
        "uv", "run", "--frozen", "python",
        "scripts/evaluation/run_ftr_202_colab.py",
        "--repo-root", ".",
        "--persistent-root", str(PERSISTENT_ROOT),
    ],
    check=True,
)


In [ ]:
import json
from pathlib import Path

generation_dir = (
    PERSISTENT_ROOT
    / SOURCE_SHA
    / "ftr-201-teacher-direct-v1"
)
handoff_path = generation_dir / "FTR_202_GENERATION_HANDOFF.json"
assert handoff_path.is_file(), handoff_path

handoff = json.loads(handoff_path.read_text())
assert handoff["source_git_sha"] == SOURCE_SHA
assert handoff["scoring_performed"] is False
assert handoff["candidate_execution_performed"] is False
print(json.dumps(handoff, indent=2, sort_keys=True))
print("\nGeneration directory to transport for isolated scoring:")
print(generation_dir)


## After generation

Copy the entire printed `ftr-201-teacher-direct-v1` directory to the isolated scoring machine. Do **not** score it in this Colab runtime because Google Drive and model-download credentials may be mounted here.

Follow `docs/FTR_202_A100_EXECUTION_HANDOFF_2026-09-10.md` for the exact Docker scoring and FTR-G2 decision procedure.
